In [1]:
import torch
torch.cuda.is_available()

True

In [5]:
import numpy as np
import random
import math
import pandas as pd
import matplotlib.pyplot as plt

In [6]:
class SlotArm():
    def __init__(self, p):
        self.p = p

    def draw(self):
        return 1 if random.random() > self.p else 0

In [7]:
class EpsilonGreedyAgent():
    def __init__(self, epsilon):
        self.epsilon = epsilon
    def initialize(self, n_arms):
        self.n = np.zeros(n_arms) # 各アームの試行回数
        self.v = np.zeros(n_arms) # 各アームの価値
    
    def select_arm(self):
        if self.epsilon > random.random():
            # ランダムにアームを選択
            return np.random.randint(0, len(self.v))
        else:
            # 価値が最大のアームを選択
            return np.argmax(self.v)
    
    def update(self, chosen_arm, reward, t):
        # 選択したアームの試行回数を更新
        self.n[chosen_arm] += 1

        # 選択したアームの価値を更新
        n= self.n[chosen_arm]
        v = self.v[chosen_arm]
        
        self.v[chosen_arm] = ((n-1)/ float(n)) * v + (1/float(n)) * reward

    # 文字列情報の取得
    def label(self):
        return "Epsilon-Greedy(" + str(self.epsilon) + ")" 

In [8]:
class UCB1():
    def initialize(self, n_arms):
        self.n = np.zeros(n_arms) # 各アームの試行回数
        self.w = np.zeros(n_arms) # 各アームの成功回数
        self.v = np.zeros(n_arms) # 各アームの価値
    def select_arm(self):
        # nがすべて1以上になるようにアームを選択
        for arm in range(len(self.n)):
            if self.n[arm] == 0:
                return arm
        # 価値が高いアームを選択
        return np.argmax(self.v)
    # アルゴリズムのパラメータを更新
    def update(self, chosen_arm, reward, t):
        # 選択したアームの試行回数を更新
        self.n[chosen_arm] += 1

        # 成功したときは選択したアームの成功回数を加算
        if reward == 1:
            self.w[chosen_arm] += 1

        # 試行回数が0のアームが存在するときは価値を更新しない
        for i in range(len(self.n)):
            if self.n[i] == 0:
                return
            
        # 各アームの価値を更新
        for i in range(len(self.n)):
            self.v[i] = (self.w[i] / self.n[i]) + math.sqrt((2 * math.log(t)) / self.n[i])

    def label(self):
        return "UCB1"